# AFM Remount Training\n\nColab notebook for retraining the remount models used by the AFM relocation workflow.\n\nThis notebook supports:\n- `train_remount_real.py`\n- `train_remount_5w.py`\n

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n

## Configure Project Path\n\nSet this to the folder that contains `afm_control_panel.py` and the `training/` directory.\n

In [ ]:
from pathlib import Path\nimport os\n\nPROJECT_ROOT = Path('/content/drive/MyDrive/AFM-Hysteresis-Simulation')\nassert PROJECT_ROOT.exists(), f'Project folder not found: {PROJECT_ROOT}'\nassert (PROJECT_ROOT / 'afm_control_panel.py').exists(), 'afm_control_panel.py not found in PROJECT_ROOT'\nos.chdir(PROJECT_ROOT)\nprint('Using project root:', PROJECT_ROOT)\n

In [ ]:
!python --version\n!nvidia-smi || true\n!pip install -r requirements.txt\n

## Sanity Checks\n\nThe remount trainers need saved site memories under `collected_data/site_memories/`.\n

In [ ]:
site_memory_root = PROJECT_ROOT / 'collected_data' / 'site_memories'\nmodels_root = PROJECT_ROOT / 'collected_data' / 'models'\nprint('site_memory_root:', site_memory_root)\nprint('exists:', site_memory_root.exists())\nif site_memory_root.exists():\n    entries = sorted([p.name for p in site_memory_root.iterdir()])\n    print('site memory entries:', len(entries))\n    print(entries[:20])\nmodels_root.mkdir(parents=True, exist_ok=True)\nprint('models_root:', models_root)\n

## Training Settings\n\nChoose mode:\n- `real`: train from real saved frame pairs\n- `5w`: train synthetic large-sample remount model\n

In [ ]:
TRAIN_MODE = 'real'\nDEVICE = 'cuda'\nSAMPLES_PER_ANCHOR = 50000\nBATCH_SIZE = 256\n\nprint({\n    'TRAIN_MODE': TRAIN_MODE,\n    'DEVICE': DEVICE,\n    'SAMPLES_PER_ANCHOR': SAMPLES_PER_ANCHOR,\n    'BATCH_SIZE': BATCH_SIZE,\n})\n

## Run Training\n

In [ ]:
import subprocess\nimport sys\n\nif TRAIN_MODE == 'real':\n    cmd = [sys.executable, 'training/train_remount_real.py', '--device', DEVICE]\nelif TRAIN_MODE == '5w':\n    cmd = [\n        sys.executable,\n        'training/train_remount_5w.py',\n        '--device', DEVICE,\n        '--samples', str(SAMPLES_PER_ANCHOR),\n        '--batch', str(BATCH_SIZE),\n    ]\nelse:\n    raise ValueError(f'Unsupported TRAIN_MODE: {TRAIN_MODE}')\n\nprint('Running:', ' '.join(cmd))\nsubprocess.run(cmd, check=True)\n

## Export Manuscript Figures\n\nThis writes publication-ready remount and site-memory figures to:\n- `manuscript/generated_figures/remount/`\n

In [ ]:
!python training/export_remount_figures.py --output-dir manuscript/generated_figures/remount\n

## Inspect Output Models\n

In [ ]:
for path in sorted(models_root.glob('*.pkl')):\n    print(path.name, f'{path.stat().st_size / 1024:.1f} KB')\n

## Inspect Exported Figures\n

In [ ]:
fig_root = PROJECT_ROOT / 'manuscript' / 'generated_figures' / 'remount'\nassert fig_root.exists(), f'Figure output folder not found: {fig_root}'\nfor path in sorted(fig_root.iterdir()):\n    print(path.name, f'{path.stat().st_size / 1024:.1f} KB')\n

## Optional: Train Legacy Deep Models\n\nThis trains `deep_same_site_classifier.pkl` and legacy `deep_remount_predictor.pkl`.\n

In [ ]:
# !python training/train_ml_models.py\n